# Pooling and Downsampling Operators: GeM, BlurPool, PixelUnshuffle, and Attention Pooling (C03 Topic 05)

**Citations:** He et al. (2014 SPP, 2017 RoIAlign); Shi et al. (2016 PixelShuffle); Radenovic et al. (2018 GeM); Zhang (2019 BlurPool); Lee et al. (2019 PMA).

### Purpose & Topic Bridges
Downsampling operators govern resolution compression, scale hierarchies, and invariance properties across deep networks. This notebook implements Generalized Mean Pooling (GeM), Anti-Aliased Downsampling (BlurPool), lossless spatial-to-channel reorganization (PixelUnshuffle), and Permutation-Invariant Attention Pooling (PMA).

## 2. Mathematical Foundations & Governing Equations

1. **Generalized Mean Pooling (GeM):** $f(x) = (\frac{1}{|R|} \sum_{u \in R} x_u^p)^{1/p}$, smoothly interpolating between Average Pooling ($p=1$) and Max Pooling ($p \to \infty$).
2. **Anti-Aliased Downsampling (BlurPool):** Factorizes strided operations into dense evaluation followed by binomial low-pass filtering $K = \frac{1}{16} [1, 2, 1]^T \otimes [1, 2, 1]$ and stride-2 subsampling, adhering to the Nyquist-Shannon sampling theorem.
3. **PixelUnshuffle:** Inverts sub-pixel convolution, losslessly trading spatial resolution for channel capacity ($H, W, C \to H/r, W/r, C \cdot r^2$) with zero FLOPs.
4. **Attention Pooling (PMA):** Uses learned seed queries $S \in \mathbb{R}^{1 \times K \times D}$ to attend over variable-length inputs $X \in \mathbb{R}^{B \times L \times D}$, producing fixed-size permutation-invariant summaries.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/05_pooling_and_downsampling_operators.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class PoolingModule(nn.Module):
    def __init__(self, p_gem, blur_kernel, s_seed):
        super().__init__()
        self.p_gem = float(p_gem[0])
        self.register_buffer('blur_kernel', blur_kernel)
        self.register_buffer('s_seed', s_seed)
        self.linear = nn.Linear(8, 4)

    def gem_pool(self, x, eps=1e-6):
        return (x.clamp(min=eps).pow(self.p_gem).mean(dim=(-2, -1))).pow(1.0 / self.p_gem)

    def blur_pool(self, x, stride=2):
        xp = F.pad(x, (1, 1, 1, 1), mode='reflect')
        return F.conv2d(xp, self.blur_kernel, stride=stride, groups=x.shape[1])

    def pixel_unshuffle(self, x, r=2):
        return F.pixel_unshuffle(x, downscale_factor=r)

    def pma(self, x):
        q = self.s_seed.expand(x.shape[0], -1, -1)
        scores = torch.matmul(q, x.transpose(-2, -1)) / math.sqrt(x.shape[-1])
        return torch.matmul(F.softmax(scores, dim=-1), x).squeeze(1)

    def forward(self, x): return self.linear(x)

model = PoolingModule(weights['p_gem'], weights['blur_kernel'], weights['s_seed']).to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('PoolingModule initialized successfully.')

In [ ]:
x_gem, x_blur, x_pixel, x_pma = weights['x_gem'], weights['x_blur'], weights['x_pixel'], weights['x_pma']

y_gem = model.gem_pool(x_gem)
err_gem = (y_gem - weights['y_gem']).abs().max().item()
assert err_gem < 1e-5, f'GeM parity error: {err_gem:.2e}'

y_blur = model.blur_pool(x_blur)
err_blur = (y_blur - weights['y_blur']).abs().max().item()
assert err_blur < 1e-5, f'BlurPool parity error: {err_blur:.2e}'

y_pixel = model.pixel_unshuffle(x_pixel)
err_pixel = (y_pixel - weights['y_pixel']).abs().max().item()
assert err_pixel < 1e-5, f'PixelUnshuffle parity error: {err_pixel:.2e}'

y_pma = model.pma(x_pma)
err_pma = (y_pma - weights['y_pma']).abs().max().item()
assert err_pma < 1e-5, f'PMA parity error: {err_pma:.2e}'

y_anchor = model(test_in)
err_anchor = (y_anchor - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor parity error: {err_anchor:.2e}'

print(f'All 5 invariants verified! GeM: {err_gem:.2e}, Blur: {err_blur:.2e}, Pixel: {err_pixel:.2e}, PMA: {err_pma:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'GeM Pooling': lambda: model.gem_pool(x_gem),
    'BlurPool Anti-Aliased': lambda: model.blur_pool(x_blur),
    'PixelUnshuffle': lambda: model.pixel_unshuffle(x_pixel),
    'PMA Attention Pooling': lambda: model.pma(x_pma),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Pathological Failure Mode: Nyquist Sampling Violation in Stride-2 Subsampling vs BlurPool Shift Equivariance
x = torch.zeros(1, 1, 8, 8, device=device)
x[0, 0, 2::2, 2::2] = 1.0 # High frequency grid pattern
x_shift = torch.roll(x, shifts=(1, 1), dims=(-2, -1)) # 1-pixel diagonal shift

# Vanilla stride-2 subsampling: complete phase cancellation
vanilla_sub = x[:, :, ::2, ::2]
vanilla_shift_sub = x_shift[:, :, ::2, ::2]
vanilla_error = (vanilla_sub - vanilla_shift_sub).abs().mean().item()

# BlurPool anti-aliased subsampling: low-pass filter suppresses aliased phase flips
k2d = torch.tensor([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=torch.float32, device=device).view(1, 1, 3, 3) / 16.0
blur_sub = F.conv2d(F.pad(x, (1, 1, 1, 1), mode='reflect'), k2d, stride=2)
blur_shift_sub = F.conv2d(F.pad(x_shift, (1, 1, 1, 1), mode='reflect'), k2d, stride=2)
blur_error = (blur_sub - blur_shift_sub).abs().mean().item()

print(f'[FAILURE DEMONSTRATED] Stride-2 sub error: {vanilla_error:.4f} vs BlurPool error: {blur_error:.4f}')
assert vanilla_error > 2.0 * blur_error, f'Expected BlurPool to improve shift equivariance by >2x'

opt_model = torch.compile(model, fullgraph=True)
assert torch.allclose(opt_model(test_in), test_out, atol=1e-5), 'Compiled model mismatch'
print('[CORRECT] Model verified with torch.compile(fullgraph=True) and torch.export.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Anti-Aliasing Guarantees:** BlurPool restores translational equivariance in deep neural networks by enforcing the Nyquist-Shannon sampling criterion before downsampling.
2. **Adaptive Continuous Saliency:** GeM pooling provides a differentiable continuum between average and max pooling, allowing retrieval models to adaptively isolate key features without hand-tuned heuristics.
3. **Zero-Loss Reorganization:** PixelUnshuffle preserves all spatial information by projecting it directly into channel dimensions, creating reversible hierarchies for multi-scale vision and audio processing.